# Module 5: Songs Data Cleaning and Analysis

Wednesday guided workshop. Work one concept at a time: predict, write a small
piece of code, inspect the result, then explain what changed.

**Question:** What can we say about the songs in this playlist after checking
and cleaning the data? This is not a representative sample of all music.

The API helper is provided setup, not exam material. Your pandas work is the
focus. Use AI for one hint or feedback on your attempt, not completed cells.
**Part A is our in-class cleaning workshop. Part B is your individual homework:**
continue with the cleaned table, summarize it, and create two plots. Submit this
same notebook, `module05_songs.ipynb`, on Gradescope. The songs assignment
replaces the Ralphs homework; the older files remain available for reference.

### Open it in VS Code

Keep this notebook in the repository's top-level folder beside `lyrics_api.py`.
In a terminal opened in `05-wrangle`, run `uv sync`. In the notebook choose
**Select Kernel → Python Environments**, then this repository's `.venv` Python.
If it is not listed, run `uv run python -m ipykernel install --user --name
dso576-module5 --display-name "DSO576 Module 5"` and select that kernel.

### Choose the data source

- The default **offline demo** uses twelve fictional song records and short original
  classroom verses. It contains deliberate whitespace/case problems and missing
  lyrics. These are not real artists, songs, or cached API responses.
- To demonstrate the API, change `USE_LIVE_API` to `True` below. Only three public
  song records are requested; maximum request time is 15 seconds each. A network
  failure stays visible as a status, never silently changes the dataset.
- Complete the core exercises using the offline demo so everyone sees the same
  rows. Then optionally repeat with real songs. Do not interpret demo word counts
  as full-song measurements or publish downloaded commercial lyrics.

## Part A In class data cleaning


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from lyrics_api import fetch_songs, search_songs, get_lyrics, load_demo_songs

USE_LIVE_API = False

# Provided sample record choices, not a claim that the API metadata is correct.
# The helper verifies title and artist. Still inspect album and duration.
playlist = [
    {"song_id": "L01", "song": "Shake It Off", "artist": "Taylor Swift", "record_id": 36962115},
    {"song_id": "L02", "song": "Hello", "artist": "Adele", "record_id": 37465951},
    {"song_id": "L03", "song": "Blinding Lights", "artist": "The Weeknd", "record_id": 37015057},
]

if USE_LIVE_API:
    songs = fetch_songs(playlist)
else:
    songs = load_demo_songs()

raw = songs.copy(deep=True)
songs[["song_id", "song", "artist", "album", "duration_seconds", "status", "source"]]


### What the helper did

`fetch_songs(playlist)` collects one record per playlist entry and constructs a
DataFrame. Failed requests remain rows. `get_lyrics(song, artist, record_id=...)`
is also available when you only need one lyrics string; it raises an error for
ambiguous matches or request failures instead of silently choosing a result.

`lyrics` is a string or missing. `status` explains why: `ok`, `no_lyrics`,
`instrumental`, `not_found`, `ambiguous`, `match_mismatch`, or `request_failed`.
`not_found` in search mode means no exact match among the returned candidates,
not proof of absence from the entire service. See `detail` for more information.
`lrclib_id` identifies the chosen source record. Neither genre nor release year
is supplied by this helper.

**Optional API demo:** uncomment the next line to inspect candidate metadata.
Choose a record only after checking the returned artist, album, and duration.
An exact title and artist alone do not guarantee the intended recording.


In [ ]:
# search_songs("Hello", "Adele")


## 1. Inspect one row

Display the first rows, column names, and data types. What does one row
represent? Which columns are identifiers rather than measurements? Display one
available lyrics string locally and compare it with its status. Do not paste
the whole lyrics into your written explanation.

Before coding: predict what your result will look like.

In [ ]:
# Your code here


**Your check / explanation:** Replace this sentence with your own observation.

## 2. Clean artist names

Keep `artist` unchanged. Create `artist_clean` by removing surrounding
whitespace and converting to lowercase. Display the raw and cleaned names side
by side. Which names should form the same group? Do not remove punctuation or
merge different artists just because their names look similar.

Before coding: predict what your result will look like.

In [ ]:
# Your code here


**Your check / explanation:** Replace this sentence with your own observation.

### Playlist dates

This separate table is **instructor-created playlist information**, not
metadata returned by LRCLIB. `added_on` is the date a song was added to our
example playlist, not its release date. Each `song_id` occurs once.
We will convert its dates now and merge it with the songs later.


In [ ]:
if USE_LIVE_API:
    playlist_info = pd.DataFrame({
        "song_id": ["L01", "L02", "L03"],
        "playlist_group": ["commute", "focus", "commute"],
        "added_on": ["2026-09-01", "2026-09-02", "2026-09-03"],
    })
else:
    playlist_info = pd.DataFrame({
        "song_id": ["D01", "D02", "D03", "D04", "D05", "D06", "D07", "D08", "D09", "D10", "D11", "D12"],
        "playlist_group": ["commute", "focus", "commute", "focus", "focus", "commute", "focus", "commute", "commute", "focus", "focus", "commute"],
        "added_on": ["2026-09-01", "2026-09-01", "2026-09-02", "2026-09-02", "2026-09-03", "2026-09-03", "2026-09-04", "2026-09-04", "2026-09-05", "2026-09-05", "2026-09-06", "2026-09-06"],
    })
playlist_info


## 3. Convert the date

In `playlist_info`, display `added_on` and its data type before conversion.
Convert `added_on` to a pandas datetime column with `pd.to_datetime`. Inspect
its type again and check for missing conversions. Explain why this date cannot
be used to study changes in music across release years. Keep this table for
the merge at the end of class.

Before coding: predict what your result will look like.

In [ ]:
# Your code here


**Your check / explanation:** Replace this sentence with your own observation.

## 4. Find unavailable lyrics

Select rows where `lyrics` is missing and display song, artist, status,
and detail. Report how many lyrics are available. Explain why no lyrics,
instrumental, and a request failure are different. Do not fill missing lyrics
with an empty string or missing word counts with zero.

Before coding: predict what your result will look like.

In [ ]:
# Your code here


**Your check / explanation:** Replace this sentence with your own observation.

## 5. Write one scalar function

Write `count_words(lyrics)`. It receives one string or missing value,
not a DataFrame row. Return `None` for missing or whitespace-only text;
otherwise return the number of whitespace-separated tokens. Use `pd.isna`,
string `split`, and `len` as needed. Punctuation attached to a token stays with
that token: this is a simple token count, not a linguistic analysis.

Test your function on `"blue train home"`, `"  blue   train  "`, `""`, and
`None`. Write your expected result for each before running the tests.

Before coding: predict what your result will look like.

In [ ]:
# Your code here


**Your check / explanation:** Replace this sentence with your own observation.

### Apply the function with map

After your function works on a single value, run the provided line below.
Each lyrics value is passed to your function; its return value becomes one
cell in the new column. Uncomment both lines after completing the function.

In [ ]:
# songs["word_count"] = songs["lyrics"].map(count_words)
# songs[["song", "word_count", "status"]]


## 6. Check the new column

For one row with lyrics, independently count tokens in a short excerpt
and check your function on that excerpt. Inspect the full column's data type
and missing values. Confirm unavailable lyrics remain missing word counts,
not zero-word songs. Keep every source row; filtering for a particular chart
comes later. Describe one limitation of comparing these text lengths.

Before coding: predict what your result will look like.

In [ ]:
# Your code here


**Your check / explanation:** Replace this sentence with your own observation.

## 7. Merge without losing songs

Merge `songs` with the `playlist_info` table whose dates you converted
earlier. Merge on `song_id` into `enriched`.
Keep every song, including those without lyrics. Check key uniqueness and show
the row counts before and after. Why would an unexpected row-count increase
be a warning? Do not join on the messy artist names.

Before coding: predict what your result will look like.

In [ ]:
# Your code here


**Your check / explanation:** Replace this sentence with your own observation.

## Before leaving class

Your table `enriched` should contain one row per song ID, the original columns,
`artist_clean`, `word_count`, `playlist_group`, and datetime `added_on`.
Keep `raw` unchanged. Save your cleaning code and explanations in this notebook.
You will reuse these cells at home; do not copy the displayed table by hand.
If a step is unfinished, complete it before starting the homework. Ask for a
hint on your attempt if you are stuck.

## Part B Homework groupby and plots

Work individually using the **same twelve-row offline dataset** from Part A.
Keep `USE_LIVE_API = False` for this assignment. You do not need to fetch songs
again or build an API client. The optional three-song live demo is too small
for the required artist comparison. Do not mix it with the fictional records.

Use pandas plotting or Matplotlib for the two required charts. A dashboard,
Streamlit, and a fitted statistical model are not required. Code cells below
are intentionally empty: predict, write your own code, and inspect the output.


## 1 Check your starting table

Continue with `enriched`, the table you cleaned in class. Display song ID,
song, `artist_clean`, `duration_seconds`, `word_count`, `status`, and
`playlist_group`. Confirm one row per song ID and the same row count as `raw`.
Report how many word counts are missing. Keep those songs in the starting table;
missing word counts are not zero.

Before coding: predict what your result will look like.

In [ ]:
# Your code here


**Your check / explanation:** Replace this sentence with your own observation.

## 2 Summarize by artist

Use `groupby` and `.agg` to create `artist_summary`, with one row per
`artist_clean` and these four columns:

- `total_songs`: number of songs, including songs without lyrics.
- `songs_with_text`: number of non-missing `word_count` values.
- `mean_words`: mean of the available word counts.
- `mean_duration_seconds`: mean duration of all songs with known duration.

Display the summary sorted from highest to lowest `mean_words`. Explain why
`total_songs` and `songs_with_text` can differ and which count is the denominator
for `mean_words`. Do not fill missing word counts before computing the mean.

Before coding: predict what your result will look like.

In [ ]:
# Your code here


**Your check / explanation:** Replace this sentence with your own observation.

## 3 Plot the artist comparison

Create a bar chart from `artist_summary` showing `mean_words` for each
artist, ordered from highest to lowest. Label the artist axis and the mean word
count axis, and give the chart an informative title. Show `songs_with_text`
beside the chart in a small displayed table or in the chart labels.

Below it, identify the artist with the largest mean in this dataset. Cite the
mean and the number of available texts supporting it. Demo texts are short
classroom verses, not full-song lyrics.

Before coding: predict what your result will look like.

In [ ]:
# Your code here


**Your check / explanation:** Replace this sentence with your own observation.

## 4 Plot individual songs

Create a scatter plot from the song-level `enriched` table, not the
artist summary. Put `duration_seconds` on the horizontal axis and `word_count`
on the vertical axis. Use only rows where both values are known; report the
number of included and excluded songs. Label both axes and title the plot.

Describe whether longer durations consistently go with larger word counts in
these records. Cite two songs that support or complicate your observation.
Do not claim a causal relationship. Demo durations are invented metadata and
the texts are short verses, so this plot cannot establish a real music trend.

Before coding: predict what your result will look like.

In [ ]:
# Your code here


**Your check / explanation:** Replace this sentence with your own observation.

## 5 Explain and check your findings

Choose one artist. Display its song-level word counts and independently
check its mean using only the available values. Show the numerator and
denominator, then compare with your summary.

Write three to five sentences answering: What did the two plots help you see?
How could missing lyrics affect the comparison? Why does this small playlist
not represent an artist's entire catalog?

End with an assistance note: whether you used Codex, one hint you received,
and how you checked your work. If you did not use it, say so.

Before coding: predict what your result will look like.

In [ ]:
# Your code here


**Your check / explanation:** Replace this sentence with your own observation.

## Finish and submit

Submit one file, `module05_songs.ipynb`, to the **Module 5 homework
assignment on Gradescope**. Include your in-class cleaning cells, homework
code, displayed summary, both plots, explanations, and assistance note.
Restart the kernel, run all cells in order, and save with outputs visible.
Keep `USE_LIVE_API = False` for the required assignment so it can run without
network access. Remove any optional outputs containing downloaded commercial
lyrics before submitting. No separate CSV, image, or PDF upload is required.
Use the posted course calendar for the deadline. Do not push student work to
the shared repository.